<a href="https://colab.research.google.com/github/x-liu/leRobot/blob/main/lerobot/training-act.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤗 x 🦾: Training ACT with LeRobot Notebook

Welcome to the **LeRobot ACT training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `ACT` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `ACT` policy for 100,000 steps typically takes **about 1.5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer.

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg, and installs the package in editable mode with train and dataset features.

In [1]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset]"

Cloning into 'lerobot'...
remote: Enumerating objects: 51899, done.
remote: Counting objects: 100% (388/388), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 51899 (delta 319), reused 229 (delta 229), pack-reused 51511 (from 3)
Receiving objects: 100% (51899/51899), 231.36 MiB | 27.13 MiB/s, done.
Resolving deltas: 100% (32960/32960), done.
Filtering content: 100% (50/50), 69.11 MiB | 12.39 MiB/s, done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

## Weights & Biases login (optional)
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging. This step is optional, you can skip it. If you want to use W&B remember to change `--wandb.enable` to true in the next section.

In [ ]:
!wandb login

## HF login

To upload your trained model to the hub you need to login with your Hugging Face account.
1. Run the cell below.
2. You will be asked to generate a token in the Hugging Face settings.
3. Select all checkboxes under Repositories when creating the token.
4. Paste the generated token into the command line below.

In [ ]:
!hf auth login

## Start training ACT with LeRobot

This cell runs `lerobot-train` to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [Action Chunking with Transformers](https://huggingface.co/docs/lerobot/act), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this. Set to `False` if you do not plan on using Weights & Biases.

7. `--batch_size=8`:  
   Increase it if you memmory allows it. It defines how many datapoints are processed at once.

8. `--steps=20000`:  
   Set for how many steps you want to train your model. 20 000 steps should work fine for a simple ACT policy.

In [ ]:
!lerobot-train \
  --dataset.repo_id=username/hf_act_record \
  --policy.type=act \
  --output_dir=outputs/train/hf_act_record0 \
  --job_name=hf_act_training_job \
  --policy.device=cuda \
  --wandb.enable=False \
  --policy.repo_id=username/hf_act_recordpolicy0 \
  --batch_size=8 \
  --steps=20000

Sometimes after training, you may notice that the model underperforms and cannot solve the task properly. Sometimes this is due to poor data quality, but sometimes the model simply needs more training. To continue training from a previously trained model, use `--policy.pretrained_path=username/path_to_model` and paste the path to the model you trained previously here.